In [ ]:
import pandas as pd
import re

# ==============================
# Caminhos
# ==============================

arquivo = r"seu_caminho\dados_alunos (2).xlsx"
saida   = r"seu_caminho\base_com_logs.xlsx"

# ==============================
# Leitura
# ==============================

df = pd.read_excel(arquivo, sheet_name="base_alunos")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

erros = []

def adicionar_erro(linha, coluna, problema, valor):
    erros.append({
        "linha": linha,
        "coluna": coluna,
        "tipo_problema": problema,
        "valor_encontrado": str(valor) if not pd.isna(valor) else "(vazio)"
    })

# ==============================
# Auditoria
# ==============================

nomes_com_sufixo = []

for idx, row in df.iterrows():
    linha_excel = idx + 2

    # aluno_id
    aluno_id = row.get("aluno_id")
    if pd.isna(aluno_id):
        adicionar_erro(linha_excel, "aluno_id", "ID do aluno ausente", aluno_id)

    # nome
    nome = row.get("nome")
    if pd.isna(nome) or str(nome).strip() == "":
        adicionar_erro(linha_excel, "nome", "Nome obrigatório ausente", nome)
    else:
        nome_str = str(nome).strip()
        if re.search(r"\s+\d+$", nome_str):
            nomes_com_sufixo.append(linha_excel)
        elif len(nome_str.split()) < 2:
            adicionar_erro(
                linha_excel,
                "nome",
                "Nome possivelmente incompleto (apenas um token)",
                nome_str
            )

    # escola
    escola = row.get("escola")
    if pd.isna(escola) or str(escola).strip() == "":
        adicionar_erro(linha_excel, "escola", "Escola obrigatória ausente", escola)

    # nota_final
    nota = pd.to_numeric(row.get("nota_final"), errors="coerce")
    if pd.isna(nota):
        adicionar_erro(linha_excel, "nota_final", "Nota ausente ou em formato inválido", row.get("nota_final"))
    elif nota < 0 or nota > 10:
        adicionar_erro(linha_excel, "nota_final", "Nota fora da faixa permitida (0 a 10)", row.get("nota_final"))

    # data_matricula
    data_original = row.get("data_matricula")
    data = pd.to_datetime(data_original, errors="coerce", dayfirst=True)
    if pd.isna(data):
        adicionar_erro(linha_excel, "data_matricula", "Data de matrícula inválida ou não reconhecida", data_original)

    # status_pagamento
    status = row.get("status_pagamento")
    if pd.isna(status) or str(status).strip() == "":
        adicionar_erro(linha_excel, "status_pagamento", "Status de pagamento ausente", status)
    else:
        status_original = str(status)
        status_normalizado = status_original.strip().lower()
        status_validos = ["pago", "pendente", "isento"]

        if status_normalizado not in status_validos:
            adicionar_erro(linha_excel, "status_pagamento", "Status de pagamento com valor não reconhecido", status_original)
        elif status_original.strip() != status_normalizado:
            adicionar_erro(linha_excel, "status_pagamento", "Status fora do padrão de capitalização esperado", status_original)

    # mensalidade
    mensalidade = pd.to_numeric(row.get("mensalidade"), errors="coerce")
    if pd.isna(mensalidade):
        adicionar_erro(linha_excel, "mensalidade", "Mensalidade ausente ou em formato inválido", row.get("mensalidade"))
    elif mensalidade < 0:
        adicionar_erro(linha_excel, "mensalidade", "Mensalidade com valor negativo", row.get("mensalidade"))

# ==============================
# Verificação de duplicidade em aluno_id
# ==============================

duplicados = df[
    df["aluno_id"].notna() &
    df["aluno_id"].duplicated(keep=False)
].copy()

if not duplicados.empty:
    for idx, row in duplicados.iterrows():
        linha_excel = idx + 2

        adicionar_erro(
            linha_excel,
            "aluno_id",
            "aluno_id duplicado na base",
            row["aluno_id"]
        )

# ==============================
# Registro consolidado de nomes com sufixo numérico
# ==============================

if nomes_com_sufixo:
    intervalo = f"linhas {nomes_com_sufixo[0]} a {nomes_com_sufixo[-1]}"
    erros.append({
        "linha": intervalo,
        "coluna": "nome",
        "tipo_problema": (
            f"{len(nomes_com_sufixo)} registros com sufixo numérico no nome "
            "(ex: 'Ana Silva 2', 'Bruno Lima 3') — possível erro de carga em massa ou base sintética"
        ),
        "valor_encontrado": f"{len(nomes_com_sufixo)} ocorrências"
    })

# ==============================
# Salvar Excel
# ==============================

log_erros = pd.DataFrame(erros)

with pd.ExcelWriter(saida, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="base_alunos", index=False)
    log_erros.to_excel(writer, sheet_name="log_erros", index=False)

print("Arquivo gerado com sucesso!")
print(f"Total de entradas no log: {len(log_erros)}")

if not log_erros.empty:
    print("\nResumo por tipo de problema:")
    print(log_erros["tipo_problema"].value_counts())

print(f"\nArquivo salvo em: {saida}")

Arquivo gerado com sucesso!
Total de entradas no log: 31

Resumo por tipo de problema:
tipo_problema
Status fora do padrão de capitalização esperado                                                                                      6
Escola obrigatória ausente                                                                                                           4
Data de matrícula inválida ou não reconhecida                                                                                        4
Nota ausente ou em formato inválido                                                                                                  4
aluno_id duplicado na base                                                                                                           4
Nota fora da faixa permitida (0 a 10)                                                                                                3
Mensalidade com valor negativo                                                           